# Full visual comparison — every stage, every pipeline

Open this **notebook** in Google Colab, connect the same Drive account as your completed comparison, and run the one code cell. A GPU is optional because this only performs inference on one image. **It reuses the saved checkpoints and does not train anything.**

The downloaded ZIP contains:

- **FULL_COMPARISON_ALL_STAGES.png** — one row per pipeline and nine columns: original image, ground-truth mask, segmentation probability map, predicted lesion mask, mask overlay, isolated lesion, actual classifier ROI, Grad-CAM heatmap and Grad-CAM overlay.
- **SLIDE_A_SEGMENTATION_COMPARISON.png** — the segmentation stages, enlarged for slides.
- **SLIDE_B_ROI_AND_GRADCAM_COMPARISON.png** — isolated lesion, ROI and both Grad-CAM views, enlarged for slides.
- One complete image per pipeline and separate PNGs for every stage.
- Classification predictions, scores, thresholds, Dice and IoU, plus exact float inputs and map arrays.

The same benign case, **ISIC_0000231**, is used in every row. Your saved Attention U-Net best-case example and the three peers from your completed comparison are preserved. Scores and overlaps are checked against the previous comparison CSV. All Grad-CAM maps explain the **melanoma class**; Attention uses its ROI branch. Each heatmap is normalised independently.

This is a selected case, not an overall ranking. Different training splits and ROI methods remain unchanged. The supplied U-Net++ notebook implements a standard U-Net, so the figure labels it **U-Net**. If a required checkpoint is missing, the cell reports that; it will not start training.


In [ ]:
#@title Generate the full visual comparison from the checkpoints already in Drive
from pathlib import Path
from datetime import datetime, timezone
import base64
import hashlib
import importlib
import sys
import tensorflow as tf
from IPython.display import display, Image as DisplayImage
from google.colab import drive, files

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
COMPARISON_ROOT = DRIVE_ROOT / 'Melanoma_BestCase_Comparison_From_Notebooks'
PREFERRED_COMPARISON = COMPARISON_ROOT / 'comparisons/20261014T185422_968354Z'
RUNTIME = Path('/content/melanoma_full_stage_visuals')
MODULE_DIR = RUNTIME / 'modules'
CASE_DIR = RUNTIME / 'case'
MODULE_DIR.mkdir(parents=True, exist_ok=True)
CASE_DIR.mkdir(parents=True, exist_ok=True)
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

EMBEDDED_FILES = {'ISIC_0000231.jpg': {'base64': '...', 'sha256': '96a88a5e2e30c76de0d4eaea3222fd38792384ec8458d8b00bf8d4503d55572b'}, 'ISIC_0000231_Segmentation.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAA8IAAALTCAAAAAD4feT7AAAQNUlEQVR4nO3d2XriyBKFUXG+ev9X5lxU2Y1tbCuFhtwRa11UTzUYKX5lIsC9LAAAAAAAAAAAAAAAAAAAAAAAAAAAAABAK/erv4AWbld/ART1r18DdrT/Xf0FUJMV+CwS5gj3L3/DQSTMAe5P/5YjSJj9yfZEf67+ApjXfd3dqH8/7b+freAzuWHIc+8dfjMi9+X2rNXb14LN2KEcXp566PDJjAyts2bsUJ4LM85OeSISZthYwRbhY0mYJ+6PlVpzpyZhvhJtEAnzxeeC/3ur1f3Zfx783fb++d15osJnXxu6Pf/X660es7c/xFyuZhXmd/flpMXxYb23GK8kYf768YMJrxalx+PYsLB8+HDvQbWtGrSPf/aHt2velmW5m9YnHBTOWiR/n7VPX8eTN2sa2C9spDlrmzv8x9xtwFeQMKd18tsf9OW/K3gF+5L2Tuzkh2lb/VWY2E98Xrix+/L8A4Pn/Mn/3aay2r7CNa2lq5rZ4Za3if3EAekne80zsZ+4ndVOdsHpX/7+JEwYDX8kYdJ4+/QHEu7G/Bcj4WYUXI2Ee6lRcI1HsRMJt2L26/HurD70W5KEuxBwURLuQcBleS7cQrGCvcnygYQ7KFYwjyRMHpekB7YkHVQceZP7jwPRQMWC3785XvsJbn8ASrvu23KcqvcQ9370pdVP913rKXY7q6pGBfcm4aIU3IWEydf6eiXhmloPdS8SLknBfUgYokmYfF5Uohj76EYkXJCCO5EwRJMwRJNwPf320f0e8QMJQzQJQzQJk8/rwlTS+olhQxKGaBKupuEi3HofLWHIJmHiNdx4PJBwMS3HueWDftP7aURBbae57SS3feBFtS14WboOs410Ka0Lbsr/X7gO/bZkFS5DwT2PgFW4hJ7Dy7J0vQNQi37fdRxnG+l4Cu5NwukU/KDjwei486ii47z+qt9A93vENcj3e81mutnDrUG/v+k01p4L51EwDyQcR8E8kjAFdbrMSRiiSThNpwWGFSQM0SQM0SQcxj6ajyScRcHrNDpOEqake5uIJRylzVyymoQhmoSTWIQHdDlYEoZoEg7SZV3ZR5cPHEo4h4J5QsIQTcIxLMJDuuyjJTyzh7cn6JdvtLlWxXmL9qbfTdpMdpsHGka1L+sy2v6fSjMSMKt5LjwhBe+n/rHssttIUn/qznF7O5K1h7z2o4uk4L3VHnIb6dkomCESnoyC91f7mEoYokl4LrUXDA4gYYgmYYgm4anYRzNKwjNRMMMkPBEFH6T0ga39xpUspQftcmUn3SoM0SQ8DYswW0iYHspeISU8i7IjxrEkDNEkPAmL8MHckeZQCmYjCUM0CU/BIny0svto34R2AvrlBXUvTiH0e5Kyk24jfS0F8yIJQzQJX8oifJqyh1rCVyo7VpxHwhdSMK+T8HUUzA4kDNEkfBmL8LmqHm8JQzQJQzQJX6Xqvm5eRY+4hGmjZsMSvkjNcZpcyYNe9vMbUys5ShnqDbxV+AIKvk69Yy9hiCbh89VbCLiQhOml3AVUwhDNt787WblFgItZhc+lYHYmYXop98KwhE9lEb5cuVNQ7po0sXLDE6rYzBd7ODNT8DRKTb2NNEST8FkswhxCwvRT6nIqYYjm3VmnKHXZZypW4TMoeC7uSDNGwRxHwhBNwrRTah8t4RPYR3MgCUM0CdNNrX20hI9nH82RJEwzxRZhCR/OIsyhvMHyUPrlaBI+jn45gY30YRTMGSR8FAVzCgnTTLVrq4QhmoQhmoTppthOWsK0U6thCdPPvVLEEj5IpSFhZt6ddQgBc5ZqH9uYgX4D1Bl8G+ndKZgzSRiiSRiiSRiiSZiO6tzNkjBkkzBEk/B+vJqUo9C5KvSc4Fr/ZuJWajoqKzP5ZR7ItWQbp8zk20jvQcFcRsL0VOayK+EdlJkGAkkYokmYpqrsnST8uiqzQCQJQzQJQzQJQzQJv67M+3xIJGGaqnLllTBNVXkhQcKvqzIL3RQ5bxJ+XZUdWTs1GpYwfZVoWMIQTcI7sJNOVWEZljCdFWhYwhBNwrSWvwxLeAf5Y0AuCUM0Cb/OIpzs/u0/hPB6yMsSTzv/eU/g/umfQ1iFX3G/K7iM1BP55+ovINf9/Qe4jlV4K/GWcn8/n2knVsI0l76bSnvuPo3gc86P0pKwCm+jYCbhdtYG+mUeabuGGSi4sLwgbKSHKbiwvIIlDNkkDO8CF2EJw7vEgiUM2SQM0SQ8LHK3xQqZZ1bCEE3C4zIv1hQlYYgmYfgrdHclYYgm4Q1CL9eUJGFYliX3wizhDXxWqaDUgiW8RezZ5lu551TCEE3CG9hIMw8Jj1MwE5HwMAUzEwlDNAlD8g1pCQ+zj2YqEh4VfL2mIglD9HVZwpBcsITHRZ9vypEw7WVflCVMd9kFSxiySZjuwl/plzDtZTcsYYgm4WHZ12yqkTBEX5UlDNEkPCr6ik09EoZoEoZoEh5kH81cJAzRJAzRJAzRn1WSMESTMESTMO1F76MlPMhrSvVkFyxhyCbhIRZhZiNhiCZhiCZhiCZhmgu/IS1hyCbhEW5IMx0JQzQJQzQJQzQJQzQJQzQJQzQJD/CaUkXpZ1XC66Wfa0qSMESTMN2F764kvFr4maYoCUM0CdNe9v5KwhBNwhBNwmtl77b4Qfb37ZAw7WVfnSW8UvZppi4JQzQJr2MRZlIShmgShmgShmgShug7HRKGaBJeJfoyza+Sz6+EIZqEIZqEIXonLWGIJmGIJmGIJmGIJmGI/sYdEoZoEoZoEoZoEl4j+IV/Vgk+wxKGaBJeI/h+JdVJGKJJGKJJeBU7aWYl4XU0zKSM5nrBLzzwm9wQrMLr5Z5lCpPwAA2XFXxqJQzRJDzAk2HmI2GIJuEBwU+Y+FnwBkvCEE3CEE3CI+yky8rdSUt4yE3ETMZIjsu9YPO92BKswrAswRdmCUM0CcOy2EgD15AwRJPwsNj7Hvwgdh8tYcgmYUhehCU8Lvhs843kcyphSC5YwpBNwrQXvQhLeJwXlZiJhCGahIdlb7uoRsIQTcJ0F76tkjBEkzBEkzBEkzBEkzBEkzBEk/Awb7BkJhKGaBKGaBKGaBKGaBIeFv6WWj4Lvz9pHseFn3KeCO7AKgzLcs+9LksYokkYokkYliX4DoeEIZqExwXfvaQeCcOyBF+YJQzRJLxB7AWbgkzjJrG3L3kuuAOr8CbBZ5wnks+nhCG5YAlDNglDNAlv4nZWJdH7aAlvomCmIeENFMw8JAzRJDzOIsxEJDxMwcxEwqMUzFQkDNEkTHfZLwtLeJR9NHOR8KDwSzblSJjm0i/KEoZoEh6VftHmo/jzGf8ALuCOVhkVxt8qPK7CeWdZipxJCdNWiYIlvEWNU08NEoZoEqarIpspCUO0P1d/AXCFIkvwImHaqRPvX9Uezym8tyPJ7fF8FZx3z4Wp7fbYbcGCJUxtt/cflpoF13xQR7ORjvHffN+rjrpVmB6qFixhyCZhKmvwnKfs9uJQDQajjuojbhWmuOrXWwlDNAlDNAlvUH1rVkzx0yVhiCbhccWv6mSRMESTMEST8DD7aGYiYcqrfdGVMESTMESTMPWV3klLmAYqNyxhiCZhOii8DEt4VOFhIJGEIZqEaaHu5knCg+qOApkkTAt1vwmehCGahCGahCGahOmg7lNhCUM2CUM0CQ8qvCMrrPJZkzBEkzBEk/CoynuyqkqfMwlDNAkPK31JJ4553MTnlZLUHvI/V38BcIBbn8ts7QvUcbrMR6h/Y31fGky4VZi6yue7LG5nbWQRnlun8yNhiCZhiCbhLTrt0zI1OkMShmgShmgShmgS3qDRE61ULV4R/kvCEE3CEE3C4+yjp9doHy3hDTrNR6ZWZ0jClNOqYAlTTq+CfdiQYpoFbBXepN2UBOl3biQM0SQM0SS8Rb/dGtOSMKX0e9+NhCGahDexk2YWEt5Gw7Nqt5OWMEST8EaW4Um1OzES3urWblaYkjl8SbsnXvNrN9HtHvDONDyN23JvOc4dH/OeJDyLtpPsufBr2g7ObPqeiL6PfD9W4gn0HWSr8Ov6Ts88Gp8DCe+g8fxMovMZkDD5OhcsYfK1Lti3vyNd74CtwqTrXrCEyda+YAkTTcESJpmCJUwyBS8S3odR4jIShmgSJpbNz7I4CnvxaaWzmdx/vDuLPPJ9YCNNHAU/kvA+TNV5HOsPJLwTc8U1JLwXDZ/Egf5IwhBNwmSxCH8iYaIo+DMJ78ZwcQUJk8R18gsJQzQJ78cKcTzvRf9CwjvSMOeT8J40fDRH+AsJ78qEcTYJQzTLxs7cbznG7W5Yn3NU9nf/O3DsyJx+y6E5iIb3ZEy/57kw81PwDyR8EFO3H8fyJxKGaBKGaBJmdvbRP5IwRJPwUawdO3EgfyZh5qbgX0gYokkYokn4MHaAe3AUfyNhZqbgX0kYokn4OFYQTiDhA2n4VY7g7yTMvBS8goSPZAQ5nIQPpeFXOHprOEqH8y14NjKbq1iFD2cSOZKEj3cTMccxXSexnR5mNlexCp/EPHIMCUM0CZ/FMjzIAVtHwqcxkhzBXJ3KTa31jOY6jtP5/nV8U/Q3bsvdXK7mUF3ibUQ1/ISZHOJwXUvDXxjJMW5nXcu88iIjdDkL8QcmcpBV+HJm9gNXtEESvp6GeYGEIZqEmYs9ySAJT8DUsp2EIdqfq78AeGBDMswhm4JXUpbFMG7jqM1CxWZxE4dtOm1bNoubuJ01HZPMCPMypfut31psFLdx3GbVrWGTuJGN9KyMNKtIeFq9Gu71aPck4Xn5H7mwgoRnpmF+ZUjm1uSmljHcznukuZp+X2IjPbcG493gIR5KwpMrP+DlH+DRJDy74iNe/OGdQMJcScEvk/D0Kk955cd2FnekuYp+d2EV5iIK3oeE51dz1ms+qgtImEsoeC8S5goK3o2EA9Sb93qP6DruSHOa27Isd/3uzOFMUOPzSmbtEA5rhMCGb/++6L8TdjdoR3FkU4RVbLDO4nZWiqwmsr7aaBKOoQqeMRdBUvbShupMVuEgIWmEfJlVSDjJWxw335+WN0YhzdvLM9Puqo3UuRzvXJNGbKTO5XhHm7FiI3UuxzvcBBG/z5D3P1/BIU93acPG53ruSLOdgifgw4Y8d3u6wH/4twqegbOQbt1G+ja64b49/b3fxsWz3nlYhYv62Oxwb7/8Av3Ow3PhdM9run35D69X5z1hU5JwSbf3Hx7+8cUA9TsnCdfzvlq+v6X601+3/a6v/GKO48Tk+3TT6eMp/fAdb9bf0vr6RmyTMimrcDU/PQVeneHXn6jgWbkjXcvvd5K3vZtLwdNyavK9R7n2ZP5e8ePbnkd+Z87n3OR7K3L9ufyt4Q+/k+8fOzfPhZt70uenV58UPDfnp4Dx3e7jr/i4JBuINM5YDeO73cdf4ZMLwZwylsVtK8g3wff/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIMX/AWHTmY5QCIzyAAAAAElFTkSuQmCC', 'sha256': 'a0540062d27cfd84a3e711f5c490ba9e6183c35ab476d8bf8dbad4549dae8796'}}

for name, record in EMBEDDED_FILES.items():
    content = base64.b64decode(record['base64'])
    if hashlib.sha256(content).hexdigest() != record['sha256']:
        raise ValueError('Embedded reference image integrity check failed: ' + name)
    (CASE_DIR / name).write_bytes(content)

MODULES = {'attention_pipeline_core.py': '"""Inference and Gradio UI for the frozen Attention U-Net melanoma candidate.\n\nNo training or threshold selection occurs in this module. JPEG ROI encoding is\npreserved because the training notebook classified saved quality-95 JPEG ROIs.\n"""\nfrom __future__ import annotations\n\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib\nimport json\nimport threading\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy import ndimage\nimport tensorflow as tf\n\nSEG_SIZE = (256, 256)\nCLS_SIZE = (224, 224)\n\n\ndef image_tensor(path, size, normalise=False):\n    pixels = tf.io.decode_image(tf.io.read_file(str(path)), channels=3,\n                                expand_animations=False)\n    pixels = tf.cast(tf.image.resize(pixels, size, method="bilinear"), tf.float32)\n    return pixels / 255.0 if normalise else pixels\n\n\ndef roi_tensor(roi):\n    # Match roi.save(..., quality=95) -> decode_classifier_image() in training.\n    buffer = BytesIO()\n    roi.convert("RGB").save(buffer, format="JPEG", quality=95)\n    pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    return tf.cast(tf.image.resize(pixels, CLS_SIZE), tf.float32)\n\n\ndef crop_from_prediction(image_pil, predicted_mask, padding=0.15):\n    image = image_pil.convert("RGB")\n    width, height = image.size\n    binary = Image.fromarray(np.uint8(np.asarray(predicted_mask) > 0) * 255)\n    binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST)) > 0\n    labelled, count = ndimage.label(binary)\n    reason = "cropped"\n    if count == 0:\n        crop, reason = image, "empty predicted mask"\n    else:\n        sizes = np.bincount(labelled.ravel())\n        sizes[0] = 0\n        lesion = labelled == int(sizes.argmax())\n        area = int(lesion.sum())\n        if area < max(64, int(width * height * 0.001)) or area > width * height * 0.80:\n            crop, reason = image, "implausible predicted area"\n        else:\n            ys, xs = np.where(lesion)\n            span = max(np.ptp(xs) + 1, np.ptp(ys) + 1)\n            margin = int(np.ceil(span * padding))\n            x0 = max(0, int(xs.min()) - margin)\n            x1 = min(width, int(xs.max()) + margin + 1)\n            y0 = max(0, int(ys.min()) - margin)\n            y1 = min(height, int(ys.max()) + margin + 1)\n            crop = image.crop((x0, y0, x1, y1))\n    a = np.asarray(crop)\n    h, w = a.shape[:2]\n    side = max(h, w)\n    top, left = (side - h) // 2, (side - w) // 2\n    square = np.pad(a, ((top, side - h - top), (left, side - w - left), (0, 0)),\n                    mode="edge")\n    roi = Image.fromarray(square).resize((CLS_SIZE[1], CLS_SIZE[0]),\n                                         Image.Resampling.BILINEAR)\n    return roi, reason\n\n\ndef mask_overlay(image, mask):\n    base = np.asarray(image.convert("RGB"), dtype=np.float32)\n    mask = Image.fromarray(np.uint8(mask) * 255).resize(\n        image.size, Image.Resampling.NEAREST)\n    binary = np.asarray(mask) > 127\n    tint = np.array([35, 195, 145], dtype=np.float32)\n    base[binary] = 0.65 * base[binary] + 0.35 * tint\n    boundary = binary & ~ndimage.binary_erosion(binary)\n    base[boundary] = [255, 215, 30]\n    return Image.fromarray(np.uint8(np.clip(base, 0, 255)))\n\n\ndef gradcam_overlay(roi, heatmap):\n    from matplotlib import colormaps\n    image = roi.convert("RGB").resize((CLS_SIZE[1], CLS_SIZE[0]))\n    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(\n        image.size, Image.Resampling.BILINEAR)\n    colors = colormaps["jet"](np.asarray(heat, dtype=np.float32) / 255.0)[..., :3]\n    base = np.asarray(image, dtype=np.float32) / 255.0\n    return Image.fromarray(np.uint8(255 * np.clip(0.62 * base + 0.38 * colors, 0, 1)))\n\n\nclass MelanomaPipeline:\n    def __init__(self, segmenter, classifier, configuration, run_dir=None):\n        self.segmenter = segmenter\n        self.classifier = classifier\n        self.configuration = dict(configuration)\n        self.run_dir = Path(run_dir) if run_dir else None\n        self.lock = threading.RLock()\n        cfg = self.configuration\n        if cfg.get("view") != "dual":\n            raise ValueError("This app requires the full-image + ROI classifier (view=\'dual\').")\n        for field in ("threshold", "segmentation_threshold"):\n            if field not in cfg or not 0 <= float(cfg[field]) <= 1:\n                raise ValueError(f"Missing or invalid saved configuration field: {field}")\n        for field in ("method", "segmentation_method"):\n            if cfg.get(field) not in ("single", "4-flip average"):\n                raise ValueError(f"Unsupported inference method: {cfg.get(field)}")\n        if segmenter.input_shape[1:] != (*SEG_SIZE, 3):\n            raise ValueError("Segmenter input must be 256 x 256 RGB.")\n        if len(classifier.inputs) != 2:\n            raise ValueError("Expected two classifier inputs: full image and ROI.")\n        shapes = [tuple(x.shape[1:]) for x in classifier.inputs]\n        if shapes != [(*CLS_SIZE, 3), (*CLS_SIZE, 3)]:\n            raise ValueError(f"Classifier input shapes are incompatible: {shapes}")\n\n    @classmethod\n    def load(cls, run_dir):\n        run_dir = Path(run_dir)\n        paths = {\n            "configuration": run_dir / "tables" / "selected_configuration.json",\n            "segmenter": run_dir / "models" / "attention_unet_best.keras",\n            "classifier": run_dir / "models" / "selected_final_classifier.keras",\n        }\n        missing = [str(p) for p in paths.values() if not p.is_file()]\n        if missing:\n            raise FileNotFoundError("The saved pipeline is incomplete. Missing:\\n" + "\\n".join(missing))\n        configuration = json.loads(paths["configuration"].read_text())\n        provenance = run_dir / "tables" / "segmentation_configuration.json"\n        if provenance.is_file():\n            expected = json.loads(provenance.read_text()).get("checkpoint_sha256")\n            if expected:\n                actual = hashlib.sha256(paths["segmenter"].read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError("The segmentation checkpoint does not match its saved configuration.")\n        manifest_path = run_dir / "deployment_manifest.json"\n        if manifest_path.is_file():\n            for relative, expected in json.loads(manifest_path.read_text()).get("sha256", {}).items():\n                actual = hashlib.sha256((run_dir / relative).read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError(f"Deployment bundle file has changed: {relative}")\n        # compile=False avoids needing custom training losses for inference.\n        segmenter = tf.keras.models.load_model(paths["segmenter"], compile=False)\n        classifier = tf.keras.models.load_model(paths["classifier"], compile=False)\n        return cls(segmenter, classifier, configuration, run_dir)\n\n    def segmentation_probabilities(self, images, batch_size=4):\n        images = np.asarray(images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "segmentation_method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            transformed = np.flip(images, axis=axes) if axes else images\n            score = self.segmenter.predict(np.ascontiguousarray(transformed),\n                                            batch_size=batch_size, verbose=0)[..., 0]\n            score = np.flip(score, axis=axes) if axes else score\n            predictions.append(score)\n        return np.mean(np.stack(predictions), axis=0)\n\n    def classifier_probabilities(self, full_images, roi_images, batch_size=8):\n        full_images = np.asarray(full_images, dtype=np.float32)\n        roi_images = np.asarray(roi_images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            full = np.flip(full_images, axis=axes) if axes else full_images\n            roi = np.flip(roi_images, axis=axes) if axes else roi_images\n            features = {"full_image": np.ascontiguousarray(full),\n                        "roi_image": np.ascontiguousarray(roi)}\n            predictions.append(self.classifier.predict(\n                features, batch_size=batch_size, verbose=0).ravel())\n        return np.mean(np.stack(predictions), axis=0)\n\n    def gradcam(self, full_batch, roi_batch, target_class):\n        model = self.classifier\n        backbone = model.get_layer("efficientnetb0")\n        gap = model.get_layer("gap")\n        with tf.GradientTape() as tape:\n            full_map = backbone(full_batch, training=False)\n            roi_map = backbone(roi_batch, training=False)\n            tape.watch(roi_map)\n            features = model.get_layer("combine_features")([gap(full_map), gap(roi_map)])\n            hidden = model.get_layer("hidden_dense")(features)\n            hidden = model.get_layer("head_dropout")(hidden, training=False)\n            output = model.get_layer("prediction")\n            logit = tf.linalg.matmul(hidden, output.kernel) + output.bias\n            score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        gradients = tape.gradient(score, roi_map)\n        if gradients is None:\n            raise RuntimeError("Grad-CAM could not trace the ROI feature map.")\n        weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)\n        heatmap = tf.nn.relu(tf.reduce_sum(roi_map * weights, axis=-1)[0])\n        return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()\n\n    def predict_file(self, path, explain=True):\n        if path is None:\n            raise ValueError("Upload a dermoscopic lesion image first.")\n        path = Path(path)\n        with Image.open(path) as handle:\n            original = handle.convert("RGB")\n        with self.lock:\n            segmentation_input = image_tensor(path, SEG_SIZE, normalise=True)[None, ...]\n            probabilities = self.segmentation_probabilities(segmentation_input)[0]\n            mask = probabilities >= float(self.configuration["segmentation_threshold"])\n            roi, reason = crop_from_prediction(\n                original, mask, padding=float(self.configuration.get("roi_padding", 0.15)))\n            full_input = image_tensor(path, CLS_SIZE)[None, ...]\n            roi_input = roi_tensor(roi)[None, ...]\n            score = float(self.classifier_probabilities(full_input, roi_input)[0])\n            predicted = int(score >= float(self.configuration["threshold"]))\n            heatmap = self.gradcam(full_input, roi_input, predicted) if explain else None\n        result = {\n            "original": original, "mask": Image.fromarray(np.uint8(mask) * 255),\n            "mask_overlay": mask_overlay(original, mask), "roi": roi,\n            "score": score, "predicted_label": predicted,\n            "label": "Melanoma" if predicted else "Benign",\n            "roi_reason": reason, "probability_map": probabilities,\n        }\n        result["gradcam"] = gradcam_overlay(roi, heatmap) if heatmap is not None else None\n        return result\n\n\ndef build_demo(pipeline, sample_paths=()):\n    import gradio as gr\n    cfg = pipeline.configuration\n\n    def preview(path):\n        if path is None:\n            return None\n        with Image.open(path) as image:\n            return image.convert("RGB")\n\n    def analyse(path):\n        try:\n            result = pipeline.predict_file(path)\n        except (ValueError, FileNotFoundError, OSError) as error:\n            raise gr.Error(str(error)) from error\n        summary = f"### Model prediction: **{result[\'label\']}**\\n"\n        summary += (f"Melanoma model score: **{result[\'score\']:.4f}**. "\n                    f"Decision threshold: {float(cfg[\'threshold\']):.4f}.\\n\\n"\n                    "The score is a model output, not a calibrated clinical probability.")\n        if result["roi_reason"] != "cropped":\n            summary += "\\n\\nThe lesion crop was unreliable; the classifier used the full image for the ROI view."\n        panels = [(result["original"], "Original image"),\n                  (result["mask"], "Predicted lesion mask"),\n                  (result["mask_overlay"], "Lesion outline"),\n                  (result["roi"], "Lesion ROI"),\n                  (result["gradcam"], "Grad-CAM on ROI")]\n        return summary, panels\n\n    def clear():\n        return None, None, "Upload a dermoscopic lesion image, then select Analyse.", []\n\n    with gr.Blocks(title="Attention U-Net melanoma research demo",\n                   delete_cache=(3600, 3600)) as demo:\n        gr.Markdown("# Melanoma image analysis\\n"\n                    "Attention U-Net lesion segmentation with EfficientNetB0 classification.\\n\\n"\n                    "**Student research prototype. Not validated for clinical diagnosis.**")\n        with gr.Tab("Analyse image"):\n            with gr.Row():\n                with gr.Column(scale=1):\n                    upload = gr.File(label="Upload dermoscopic image (JPG or PNG)",\n                                     file_types=[".jpg", ".jpeg", ".png"], type="filepath")\n                    input_preview = gr.Image(label="Uploaded image", interactive=False)\n                    with gr.Row():\n                        analyse_button = gr.Button("Analyse", variant="primary")\n                        clear_button = gr.Button("Clear")\n                    if sample_paths:\n                        gr.Examples(examples=[[str(p)] for p in sample_paths],\n                                    inputs=upload, label="Public ISIC example images")\n                with gr.Column(scale=2):\n                    summary = gr.Markdown("Upload a dermoscopic lesion image, then select Analyse.")\n                    panels = gr.Gallery(label="Pipeline outputs", columns=3,\n                                        object_fit="contain", height="auto")\n                    gr.Markdown("The lesion mask estimates the lesion boundary. "\n                                "Grad-CAM shows image regions contributing to the model prediction; "\n                                "it is not a lesion mask.")\n            upload.change(preview, inputs=upload, outputs=input_preview)\n            analyse_button.click(analyse, inputs=upload, outputs=[summary, panels],\n                                 api_name="analyse", concurrency_limit=1)\n            clear_button.click(clear, outputs=[upload, input_preview, summary, panels])\n        with gr.Tab("Evaluation"):\n            gr.Markdown("## Saved evaluation results\\n"\n                        "Classification and segmentation measure different tasks. "\n                        "The official ISIC test set was viewed in earlier experiments; "\n                        "these results are exploratory follow-up.")\n            run_dir = pipeline.run_dir\n            displayed = False\n            if run_dir:\n                for filename, label in [("complete_segmentation_test_metrics.csv", "Lesion segmentation"),\n                                        ("complete_classification_test_metrics.csv", "Melanoma classification")]:\n                    path = run_dir / "tables" / filename\n                    if path.is_file():\n                        gr.Markdown("### " + label)\n                        gr.Dataframe(value=pd.read_csv(path).T.reset_index(),\n                                     headers=["Metric", "Value"], interactive=False)\n                        displayed = True\n                figures = sorted((run_dir / "figures").glob("*.png"))\n                if figures:\n                    gr.Gallery(value=[(str(p), p.stem.replace("_", " ")) for p in figures],\n                               label="Evaluation figures", columns=1, object_fit="contain")\n                    displayed = True\n            if not displayed:\n                gr.Markdown("Run the reporting notebook to add test loss, metrics and evaluation figures.")\n    return demo\n', 'bestcase_comparison_helpers.py': '\nfrom pathlib import Path\nfrom io import BytesIO\nimport gc\nimport json\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport tensorflow as tf\nimport cv2\nfrom attention_pipeline_core import MelanomaPipeline\n\nIMG_SIZE = 256\nCLS_IMG_SIZE = 224\nMASK_MODE = "mask_crop"\nSEG_IMAGE_SIZE = (256, 256)\nSEG_THRESHOLD = 0.50\n\n\ndef get_lesion_region(image, unet_model, pad_frac=0.15, mode=MASK_MODE):\n    """Runs the trained U-Net on a full-res image to get a lesion mask, then applies\n    pixel-masking and/or cropping (per `mode`) before resizing for the classifier."""\n    h, w = image.shape[0], image.shape[1]\n    seg_input = tf.image.resize(image, [IMG_SIZE, IMG_SIZE]) / 255.0\n    seg_input = tf.expand_dims(seg_input, 0)\n    pred_mask_small = unet_model(seg_input, training=False)[0, ..., 0].numpy()\n    pred_mask_small = (pred_mask_small > 0.5).astype(np.uint8)\n\n    # Upsample the mask back to the original image resolution for pixel-accurate masking\n    pred_mask_full = tf.image.resize(\n        pred_mask_small[..., np.newaxis], [h, w], method="nearest"\n    ).numpy()[..., 0].astype(np.uint8)\n\n    ys, xs = np.where(pred_mask_small > 0)\n    lesion_found = len(xs) > 0 and len(ys) > 0\n\n    work_image = image.numpy() if hasattr(image, "numpy") else np.array(image)\n\n    if mode in ("mask", "mask_crop"):\n        if lesion_found:\n            work_image = work_image * pred_mask_full[..., np.newaxis]\n        # if no lesion detected, fall back to the unmasked image rather than an all-black frame\n\n    if mode in ("crop", "mask_crop") and lesion_found:\n        x_min, x_max = xs.min() / IMG_SIZE, xs.max() / IMG_SIZE\n        y_min, y_max = ys.min() / IMG_SIZE, ys.max() / IMG_SIZE\n        pad_x = (x_max - x_min) * pad_frac\n        pad_y = (y_max - y_min) * pad_frac\n        x_min = max(0.0, x_min - pad_x); x_max = min(1.0, x_max + pad_x)\n        y_min = max(0.0, y_min - pad_y); y_max = min(1.0, y_max + pad_y)\n\n        x1, x2 = int(x_min * w), int(x_max * w)\n        y1, y2 = int(y_min * h), int(y_max * h)\n        work_image = work_image[y1:max(y2, y1 + 1), x1:max(x2, x1 + 1), :]\n\n    out = tf.image.resize(work_image, [CLS_IMG_SIZE, CLS_IMG_SIZE])\n    return out.numpy()\n\ndef predict_mask_for_pil(model, image_pil):\n    resized = image_pil.resize(\n        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.BILINEAR\n    )\n    x = np.asarray(resized, dtype=np.float32) / 255.0\n    probability = model.predict(x[None, ...], verbose=0)[0, ..., 0]\n    binary = (probability >= SEG_THRESHOLD).astype(np.uint8)\n    return probability, binary\n\ndef extract_roi_from_binary_mask(image_pil, binary_mask, padding_fraction=0.10):\n    image_array = np.asarray(image_pil.convert(\'RGB\'))\n    h, w = image_array.shape[:2]\n    mask = np.asarray(\n        Image.fromarray((binary_mask * 255).astype(np.uint8)).resize(\n            (w, h), Image.Resampling.NEAREST\n        )\n    ) > 0\n\n    ys, xs = np.where(mask)\n    if len(xs) == 0 or len(ys) == 0:\n        return image_pil.copy(), (0, 0, w, h), True\n\n    x_min, x_max = int(xs.min()), int(xs.max())\n    y_min, y_max = int(ys.min()), int(ys.max())\n    box_w, box_h = max(1, x_max - x_min + 1), max(1, y_max - y_min + 1)\n    pad_x = int(round(box_w * padding_fraction))\n    pad_y = int(round(box_h * padding_fraction))\n    x0, y0 = max(0, x_min - pad_x), max(0, y_min - pad_y)\n    x1, y1 = min(w, x_max + pad_x + 1), min(h, y_max + pad_y + 1)\n\n    if (x1 - x0) < 8 or (y1 - y0) < 8:\n        return image_pil.copy(), (0, 0, w, h), True\n\n    return image_pil.crop((x0, y0, x1, y1)), (x0, y0, x1, y1), False\n\n\ndef hard_overlap(truth, prediction):\n    truth = np.asarray(truth, dtype=bool)\n    prediction = np.asarray(prediction, dtype=bool)\n    if truth.shape != prediction.shape:\n        raise ValueError(f"Mask shapes differ: {truth.shape} and {prediction.shape}")\n    intersection = int(np.logical_and(truth, prediction).sum())\n    total = int(truth.sum()) + int(prediction.sum())\n    union = int(np.logical_or(truth, prediction).sum())\n    return {\n        "dice": (2.0 * intersection / total) if total else 1.0,\n        "iou": (intersection / union) if union else 1.0,\n    }\n\n\ndef load_pair(segmenter_path, classifier_path):\n    seg = tf.keras.models.load_model(str(segmenter_path), compile=False)\n    cls = tf.keras.models.load_model(str(classifier_path), compile=False)\n    if tuple(seg.input_shape[1:]) != (256, 256, 3):\n        raise ValueError("Expected a 256 x 256 RGB segmenter.")\n    if len(cls.inputs) != 1 or tuple(cls.input_shape[1:]) != (224, 224, 3):\n        raise ValueError("Expected a single-input 224 x 224 RGB classifier.")\n    return seg, cls\n\n\ndef check_score(score):\n    score = float(score)\n    if not np.isfinite(score) or not 0 <= score <= 1:\n        raise ValueError(f"Invalid sigmoid model score: {score}")\n    return score\n\n\ndef infer_attention(image_path, run_dir):\n    pipeline = MelanomaPipeline.load(run_dir)\n    result = pipeline.predict_file(image_path, explain=False)\n    return {\n        "mask": np.asarray(result["mask"]) > 0,\n        "roi": result["roi"].copy(),\n        "score": check_score(result["score"]),\n        "threshold": float(pipeline.configuration["threshold"]),\n        "mask_threshold": float(pipeline.configuration["segmentation_threshold"]),\n        "mask_method": pipeline.configuration["segmentation_method"],\n        "classifier_method": pipeline.configuration["method"],\n        "roi_note": result["roi_reason"],\n    }\n\n\ndef infer_uploaded_unet(image_path, segmenter_path, classifier_path):\n    segmenter, classifier = load_pair(segmenter_path, classifier_path)\n    rgb = tf.cast(tf.io.decode_jpeg(tf.io.read_file(str(image_path)), channels=3),\n                  tf.float32)\n    seg_input = tf.image.resize(rgb, (256, 256)) / 255.0\n    probability = segmenter(seg_input[None, ...], training=False)[0, ..., 0].numpy()\n    # The supplied notebook uses > 0.5 for its ROI mask, then >= 0.5 for classification.\n    mask = probability > 0.5\n    roi_pixels = get_lesion_region(rgb, segmenter, pad_frac=0.15, mode="mask_crop")\n    score = check_score(classifier.predict(roi_pixels[None, ...], verbose=0).ravel()[0])\n    return {\n        "mask": mask,\n        "roi": Image.fromarray(np.clip(roi_pixels, 0, 255).astype(np.uint8)),\n        "score": score, "threshold": 0.5, "mask_threshold": 0.5,\n        "mask_method": "single", "classifier_method": "single",\n        "roi_note": "background masking + 15% bounding-box crop",\n    }\n\n\ndef infer_resunet(image_path, segmenter_path, classifier_path):\n    segmenter, classifier = load_pair(segmenter_path, classifier_path)\n    with Image.open(image_path) as image:\n        original = image.convert("RGB")\n    _, mask = predict_mask_for_pil(segmenter, original)\n    roi, bbox, fallback = extract_roi_from_binary_mask(original, mask, padding_fraction=0.10)\n    # Preserve the original saved quality-95 JPEG ROI preprocessing.\n    buffer = BytesIO()\n    roi.save(buffer, format="JPEG", quality=95)\n    roi_pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    roi_pixels = tf.cast(tf.image.resize(roi_pixels, (224, 224), method="bilinear"),\n                         tf.float32)\n    score = check_score(classifier.predict(roi_pixels[None, ...], verbose=0).ravel()[0])\n    return {\n        "mask": mask.astype(bool), "roi": roi.copy(),\n        "score": score, "threshold": 0.5, "mask_threshold": 0.5,\n        "mask_method": "single", "classifier_method": "single",\n        "roi_note": "full-image fallback" if fallback else "10% RGB bounding-box crop",\n    }\n\n\ndef infer_transunet(image_path, segmenter_path, classifier_path):\n    segmenter, classifier = load_pair(segmenter_path, classifier_path)\n    original = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n    if original is None:\n        raise ValueError("OpenCV could not decode the JPEG.")\n    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)\n    # Match the supplied notebook\'s OpenCV resize and normalisation exactly.\n    seg_pixels = cv2.resize(original, (256, 256)).astype(np.float32) / 255.0\n    probability = segmenter.predict(seg_pixels[None, ...], verbose=0)[0, ..., 0]\n    mask = probability >= 0.5\n    masked_pixels = seg_pixels * mask[..., None].astype(np.float32)\n    roi_pixels = tf.image.resize(masked_pixels, (224, 224)).numpy() * 255.0\n    score = check_score(classifier.predict(roi_pixels[None, ...], verbose=0).ravel()[0])\n    return {\n        "mask": mask,\n        "roi": Image.fromarray(np.clip(roi_pixels, 0, 255).astype(np.uint8)),\n        "score": score, "threshold": 0.5, "mask_threshold": 0.5,\n        "mask_method": "single", "classifier_method": "single",\n        "roi_note": "256-pixel background-masked full frame, resized to 224",\n    }\n\n\ndef comparison_outputs(image_path, mask_path, output_dir, models, true_label="Benign"):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    with Image.open(mask_path) as image:\n        # One explicitly shared reference and hard Dice/IoU formula for this figure.\n        truth = np.asarray(image.convert("L").resize((256, 256),\n                           Image.Resampling.NEAREST)) > 127\n    outputs = []\n    rows = []\n    order = ["Attention U-Net", "U-Net (uploaded notebook)", "ResUNet", "TransUNet"]\n    for name in order:\n        tf.keras.backend.clear_session()\n        gc.collect()\n        spec = models[name]\n        print(f"Running {name} on {Path(image_path).name}...")\n        if name == "Attention U-Net":\n            result = infer_attention(image_path, spec["run_dir"])\n        else:\n            function = {\n                "U-Net (uploaded notebook)": infer_uploaded_unet,\n                "ResUNet": infer_resunet,\n                "TransUNet": infer_transunet,\n            }[name]\n            result = function(image_path, spec["segmenter"], spec["classifier"])\n        metrics = hard_overlap(truth, result["mask"])\n        label = "Melanoma" if result["score"] >= result["threshold"] else "Benign"\n        result.update(name=name, prediction=label, **metrics)\n        slug = {"Attention U-Net": "attention", "U-Net (uploaded notebook)": "unet",\n                "ResUNet": "resunet", "TransUNet": "transunet"}[name]\n        Image.fromarray(result["mask"].astype(np.uint8) * 255).save(output_dir / f"{slug}_mask.png")\n        result["roi"].save(output_dir / f"{slug}_roi.png")\n        outputs.append(result)\n        rows.append({\n            "Pipeline": name, "Image ID": Path(image_path).stem, "True label": true_label,\n            "Prediction": label, "Model score": result["score"],\n            "Decision threshold": result["threshold"], "Hard Dice (this image)": metrics["dice"],\n            "Hard IoU (this image)": metrics["iou"], "Mask threshold": result["mask_threshold"],\n            "Mask inference": result["mask_method"],\n            "Classifier inference": result["classifier_method"], "ROI treatment": result["roi_note"],\n        })\n        tf.keras.backend.clear_session()\n        gc.collect()\n    table = pd.DataFrame(rows)\n    table.to_csv(output_dir / "single_image_comparison.csv", index=False)\n    return truth, outputs, table\n\n\ndef save_comparison_figures(image_path, truth, outputs, output_dir):\n    import matplotlib.pyplot as plt\n    output_dir = Path(output_dir)\n    with Image.open(image_path) as image:\n        original = image.convert("RGB").copy()\n    display_rgb = np.asarray(original.resize((256, 256), Image.Resampling.BILINEAR))\n    fig, axes = plt.subplots(1, 6, figsize=(19.5, 5.2), facecolor="white")\n    axes[0].imshow(display_rgb)\n    axes[0].set_title("Original image", fontsize=17)\n    axes[1].imshow(truth, cmap="gray", vmin=0, vmax=1)\n    axes[1].set_title("Expert mask", fontsize=17)\n    for ax, result in zip(axes[2:], outputs):\n        name = "U-Net*" if result["name"].startswith("U-Net (") else result["name"]\n        ax.imshow(result["mask"], cmap="gray", vmin=0, vmax=1)\n        ax.set_title(name, fontsize=17, fontweight="bold")\n        ax.text(0.5, -0.08,\n                f\'Dice {result["dice"]:.3f}\\n{result["prediction"]}\\n\'\n                f\'Score {result["score"]:.3f} / cutoff {result["threshold"]:.3f}\',\n                ha="center", va="top", transform=ax.transAxes, fontsize=13)\n    for ax in axes:\n        ax.axis("off")\n    fig.suptitle(f"Best Attention U-Net segmentation example: {Path(image_path).stem} (Benign)", fontsize=23,\n                 fontweight="bold", y=0.97)\n    fig.subplots_adjust(left=0.015, right=0.985, top=0.80, bottom=0.27, wspace=0.12)\n    fig.text(0.5, 0.06,\n             "Selected best Attention segmentation case. Peer weights may be newly trained. "\n             "*The uploaded U-Net++ notebook implements a standard U-Net.",\n             ha="center", fontsize=12)\n    fig.savefig(output_dir / "Slide1_Shared_Image_Comparison.png", dpi=200,\n                facecolor="white", bbox_inches="tight")\n    plt.show()\n\n    fig, axes = plt.subplots(4, 3, figsize=(10, 14), facecolor="white")\n    for i, result in enumerate(outputs):\n        axes[i, 0].imshow(result["mask"], cmap="gray", vmin=0, vmax=1)\n        axes[i, 1].imshow(display_rgb)\n        if result["mask"].any() and not result["mask"].all():\n            axes[i, 1].contour(result["mask"].astype(float), levels=[0.5],\n                               colors=["#00dd77"], linewidths=1.2)\n        axes[i, 2].imshow(result["roi"])\n        axes[i, 0].set_title(f\'{result["name"]}\\nDice {result["dice"]:.4f}\', fontsize=13)\n        axes[i, 1].set_title("Predicted boundary", fontsize=13)\n        axes[i, 2].set_title(f\'Classifier ROI\\nPrediction: {result["prediction"]}\', fontsize=13)\n    for ax in axes.flat:\n        ax.axis("off")\n    fig.suptitle(f"Pipeline details for {Path(image_path).stem} (true label: Benign)", fontsize=16)\n    fig.tight_layout(rect=(0, 0, 1, 0.97))\n    fig.savefig(output_dir / "Backup_Masks_Boundaries_and_ROIs.png", dpi=180,\n                facecolor="white", bbox_inches="tight")\n    plt.close(fig)\n', 'full_stage_visuals.py': '"""Export real inference stages from four existing melanoma pipelines. No training."""\nfrom pathlib import Path\nfrom io import BytesIO\nimport gc\nimport json\nimport zipfile\nfrom datetime import datetime, timezone\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport tensorflow as tf\nimport cv2\nimport matplotlib.pyplot as plt\nfrom scipy import ndimage\n\nfrom attention_pipeline_core import MelanomaPipeline, image_tensor, roi_tensor\nfrom bestcase_comparison_helpers import (\n    load_pair, get_lesion_region, predict_mask_for_pil,\n    extract_roi_from_binary_mask, hard_overlap, check_score,\n)\n\nORDER = [\'Attention U-Net\', \'U-Net (uploaded notebook)\', \'ResUNet\', \'TransUNet\']\nSLUGS = dict(zip(ORDER, [\'attention_unet\', \'unet\', \'resunet\', \'transunet\']))\nTITLES = [\'Original image\', \'Ground-truth mask\', \'Segmentation probability\',\n          \'Predicted lesion mask\', \'Mask overlay\', \'Isolated lesion\',\n          \'Classifier lesion ROI\', \'Grad-CAM heatmap\', \'Grad-CAM on ROI\']\nKEYS = [\'original\', \'ground_truth\', \'probability\', \'mask\', \'overlay\',\n        \'isolated\', \'roi\', \'heatmap\', \'gradcam\']\n\n\ndef array_rgb(pixels):\n    return np.uint8(np.clip(np.asarray(pixels), 0, 255))\n\n\ndef signed_logit_gradcam(model, image_batch, target_class=1):\n    """Follow each saved classifier\'s augmentation, backbone and head exactly.\n\n    All panels explain the melanoma logit, giving the same target across models.\n    Explicit input watching also supports fully frozen EfficientNet backbones.\n    """\n    candidates = [layer for layer in model.layers\n                  if isinstance(layer, tf.keras.Model)\n                  and \'efficientnet\' in layer.name.lower()\n                  and len(layer.output_shape) == 4]\n    if len(candidates) != 1:\n        raise ValueError(f\'{model.name}: expected one nested spatial backbone; \'\n                         f\'found {[layer.name for layer in candidates]}\')\n    backbone = candidates[0]\n    index = model.layers.index(backbone)\n    final = model.layers[-1]\n    if not isinstance(final, tf.keras.layers.Dense) or final.units != 1:\n        raise ValueError(\'Expected the saved one-unit sigmoid classification head.\')\n    batch = tf.convert_to_tensor(image_batch, dtype=tf.float32)\n    with tf.GradientTape() as tape:\n        tape.watch(batch)\n        value = batch\n        for layer in model.layers[:index]:\n            if not isinstance(layer, tf.keras.layers.InputLayer):\n                value = layer(value, training=False)\n        feature = backbone(value, training=False)\n        tape.watch(feature)\n        value = feature\n        for layer in model.layers[index + 1:-1]:\n            value = layer(value, training=False)\n        logit = tf.linalg.matmul(value, final.kernel)\n        if final.use_bias:\n            logit = logit + final.bias\n        score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        probability = tf.math.sigmoid(logit)[0, 0]\n    gradient = tape.gradient(score, feature)\n    if gradient is None:\n        raise RuntimeError(f\'{model.name}: Grad-CAM gradient could not be traced.\')\n    pooled = tf.reduce_mean(gradient, axis=(1, 2), keepdims=True)\n    heat = tf.nn.relu(tf.reduce_sum(feature * pooled, axis=-1)[0])\n    heat = heat / (tf.reduce_max(heat) + 1e-8)\n    return heat.numpy(), float(probability.numpy())\n\n\ndef detailed_attention(image_path, spec):\n    pipeline = MelanomaPipeline.load(spec[\'run_dir\'])\n    result = pipeline.predict_file(image_path, explain=False)\n    full = image_tensor(image_path, (224, 224))[None, ...]\n    roi = roi_tensor(result[\'roi\'])[None, ...]\n    heat = pipeline.gradcam(full, roi, target_class=1)\n    return dict(probability=result[\'probability_map\'],\n                mask=np.asarray(result[\'mask\']) > 0,\n                roi_pixels=roi.numpy()[0], heatmap=heat,\n                score=check_score(result[\'score\']),\n                threshold=float(pipeline.configuration[\'threshold\']),\n                mask_threshold=float(pipeline.configuration[\'segmentation_threshold\']),\n                mask_method=pipeline.configuration[\'segmentation_method\'],\n                classifier_method=pipeline.configuration[\'method\'],\n                roi_note=result[\'roi_reason\'],\n                full_classifier_input=full.numpy()[0])\n\n\ndef detailed_peer(name, image_path, spec):\n    segmenter, classifier = load_pair(spec[\'segmenter\'], spec[\'classifier\'])\n    with Image.open(image_path) as handle:\n        original = handle.convert(\'RGB\')\n    if name.startswith(\'U-Net (\'):\n        rgb = tf.cast(tf.io.decode_jpeg(tf.io.read_file(str(image_path)), channels=3),\n                      tf.float32)\n        seg_input = tf.image.resize(rgb, (256, 256)) / 255.0\n        probability = segmenter(seg_input[None, ...], training=False)[0, ..., 0].numpy()\n        mask = probability > 0.5\n        roi_pixels = get_lesion_region(rgb, segmenter, pad_frac=0.15, mode=\'mask_crop\')\n        note = \'Background masking and 15% bounding-box crop\'\n    elif name == \'ResUNet\':\n        probability, mask = predict_mask_for_pil(segmenter, original)\n        roi, _, fallback = extract_roi_from_binary_mask(original, mask, padding_fraction=0.10)\n        buffer = BytesIO()\n        roi.save(buffer, format=\'JPEG\', quality=95)\n        pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n        roi_pixels = tf.cast(tf.image.resize(pixels, (224, 224), method=\'bilinear\'),\n                             tf.float32).numpy()\n        note = \'Full-image fallback\' if fallback else \'10% RGB bounding-box crop, quality-95 JPEG\'\n    elif name == \'TransUNet\':\n        pixels = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n        if pixels is None:\n            raise ValueError(\'OpenCV could not decode the original image.\')\n        pixels = cv2.cvtColor(pixels, cv2.COLOR_BGR2RGB)\n        seg_pixels = cv2.resize(pixels, (256, 256)).astype(np.float32) / 255.0\n        probability = segmenter.predict(seg_pixels[None, ...], verbose=0)[0, ..., 0]\n        mask = probability >= 0.5\n        masked = seg_pixels * mask[..., None].astype(np.float32)\n        roi_pixels = tf.image.resize(masked, (224, 224)).numpy() * 255.0\n        note = \'Background-masked 256-pixel full frame resized to 224\'\n    else:\n        raise ValueError(name)\n    batch = np.asarray(roi_pixels, dtype=np.float32)[None, ...]\n    score = check_score(classifier.predict(batch, verbose=0).ravel()[0])\n    heat, reconstructed_score = signed_logit_gradcam(classifier, batch, target_class=1)\n    if not np.isclose(score, reconstructed_score, atol=2e-5, rtol=2e-5):\n        raise RuntimeError(f\'{name}: Grad-CAM head does not reproduce the saved \'\n                           f\'classifier score ({reconstructed_score} vs {score}).\')\n    return dict(probability=probability, mask=np.asarray(mask, dtype=bool),\n                roi_pixels=roi_pixels, heatmap=heat, score=score, threshold=0.5,\n                mask_threshold=0.5, mask_method=\'single\', classifier_method=\'single\',\n                roi_note=note, full_classifier_input=None)\n\n\ndef stage_panels(original, truth, result):\n    rgb = np.asarray(original.resize((256, 256), Image.Resampling.BILINEAR))\n    mask = np.asarray(result[\'mask\'], dtype=bool)\n    if mask.shape != (256, 256) or np.shape(result[\'probability\']) != (256, 256):\n        raise ValueError(\'Expected a 256 x 256 segmentation map.\')\n    overlay = rgb.astype(np.float32).copy()\n    overlay[mask] = .65 * overlay[mask] + .35 * np.array([35, 195, 145])\n    border = mask & ~ndimage.binary_erosion(mask)\n    overlay[border] = [255, 215, 30]\n    roi = array_rgb(result[\'roi_pixels\'])\n    raw_heat = np.asarray(result[\'heatmap\'], dtype=np.float32)\n    if not np.isfinite(raw_heat).all() or raw_heat.ndim != 2:\n        raise ValueError(\'Grad-CAM must be a finite spatial map.\')\n    heat = tf.image.resize(raw_heat[None, ..., None], roi.shape[:2],\n                           method=\'bilinear\')[0, ..., 0].numpy()\n    heat = np.clip(heat, 0, 1)\n    colors = plt.colormaps[\'jet\'](heat)[..., :3]\n    cam = .62 * roi.astype(np.float32) + .38 * colors * 255\n    # Do not paint an apparent focus when every positive contribution is zero.\n    if float(raw_heat.max()) <= 1e-8:\n        cam = roi.copy()\n    return dict(original=rgb, ground_truth=np.uint8(truth) * 255,\n                probability=np.asarray(result[\'probability\'], dtype=np.float32),\n                mask=np.uint8(mask) * 255, overlay=array_rgb(overlay),\n                isolated=rgb * mask[..., None], roi=roi,\n                heatmap=heat, gradcam=array_rgb(cam))\n\n\ndef plot_grid(results, output, filename, keys=KEYS):\n    columns = len(keys)\n    width = 3.0 * columns + 2.7\n    fig, axes = plt.subplots(len(results), columns, figsize=(width, 2.9 * len(results) + 1.9),\n                             squeeze=False, facecolor=\'white\')\n    for row, result in enumerate(results):\n        for col, key in enumerate(keys):\n            ax = axes[row, col]\n            if key in (\'probability\', \'heatmap\'):\n                ax.imshow(result[\'panels\'][key], cmap=\'viridis\' if key == \'probability\'\n                          else \'jet\', vmin=0, vmax=1)\n            elif key in (\'ground_truth\', \'mask\'):\n                ax.imshow(result[\'panels\'][key], cmap=\'gray\', vmin=0, vmax=255)\n            else:\n                ax.imshow(result[\'panels\'][key])\n            ax.axis(\'off\')\n            if row == 0:\n                title = TITLES[KEYS.index(key)]\n                ax.set_title(title, fontsize=13, pad=14, fontweight=\'bold\')\n            if key in (\'heatmap\', \'gradcam\') and result[\'no_positive_cam\']:\n                ax.text(.5, -.035, \'No positive activation\', ha=\'center\', va=\'top\',\n                        transform=ax.transAxes, fontsize=10)\n        name = \'U-Net*\' if result[\'name\'].startswith(\'U-Net (\') else result[\'name\']\n        label = (f\'{name}\\nDice {result["dice"]:.4f}\\nIoU {result["iou"]:.4f}\\n\\n\'\n                 f\'Prediction: {result["prediction"]}\\nScore {result["score"]:.4f}\\n\'\n                 f\'Cutoff {result["threshold"]:.4f}\')\n        axes[row, 0].text(-.075, .5, label, transform=axes[row, 0].transAxes,\n                         ha=\'right\', va=\'center\', fontsize=11, linespacing=1.4)\n    fig.suptitle(\'ISIC_0000231 | Actual label: Benign | All pipeline stages\',\n                 fontsize=22, fontweight=\'bold\', y=.985)\n    fig.text(.52, .035, \'Same selected image in every row. Grad-CAM target: melanoma class \'\n             \'(Attention: ROI branch). Maps are individually normalised; colours are not comparable in magnitude.\',\n             ha=\'center\', fontsize=11)\n    fig.text(.52, .012, \'Selected best Attention segmentation case; different training splits and ROI methods. \'\n             \'*The uploaded U-Net++ notebook implements U-Net. This is not an overall model ranking.\',\n             ha=\'center\', fontsize=11)\n    fig.subplots_adjust(left=2.5 / width, right=.992, top=.91, bottom=.08,\n                         hspace=.25, wspace=.08)\n    fig.savefig(output / filename, dpi=220, bbox_inches=\'tight\', facecolor=\'white\')\n    plt.close(fig)\n\n\ndef resolve_saved_models(drive_root, comparison_root, preferred_comparison=None):\n    """Resolve the checkpoints belonging to the completed comparison, without fitting."""\n    drive_root, comparison_root = Path(drive_root), Path(comparison_root)\n    preferred = Path(preferred_comparison) if preferred_comparison else None\n    manifests = ([preferred / \'comparison_provenance.json\'] if preferred else [])\n    manifests += sorted((comparison_root / \'comparisons\').glob(\'*/comparison_provenance.json\'),\n                        key=lambda p: p.parent.name, reverse=True)\n    manifests = [p for p in manifests if p.is_file()]\n    if not manifests:\n        raise FileNotFoundError(\'The completed comparison folder was not found. \'\n                                \'Use the same Drive account as your completed run.\')\n    manifest_path = manifests[0]\n    manifest = json.loads(manifest_path.read_text())\n    models = {\'Attention U-Net\': {\'run_dir\': Path(manifest[\'attention_saved_run\'])}}\n    originals = {\n        \'U-Net (uploaded notebook)\': (\n            drive_root / \'Melanoma_Comparison_Models/unet_segmenter.keras\',\n            drive_root / \'Melanoma_Comparison_Models/unet_classifier.keras\'),\n        \'ResUNet\': (\n            drive_root / \'Melanoma_AI_ResUNet_Targeted_Malignant_Aug_Results/models/resunet_best.keras\',\n            drive_root / \'Melanoma_AI_ResUNet_Targeted_Malignant_Aug_Results/models/efficientnet_b0_final.keras\'),\n        \'TransUNet\': (drive_root / \'43022/TransUNet_ISIC2016.keras\',\n                     drive_root / \'43022/EfficientNetB0_Augmented_Final.keras\'),\n    }\n    records = {row[\'Pipeline\']: row for row in manifest[\'peer_model_provenance\']}\n    missing = []\n    for name in ORDER[1:]:\n        if name not in records:\n            raise ValueError(\'Comparison provenance is missing \' + name)\n        record = records[name]\n        folder = comparison_root / \'trained_models\' / record[\'Code hash\'] / name.split()[0].lower()\n        cached = (folder / \'selected_segmenter.keras\', folder / \'selected_classifier.keras\')\n        if \'NEW\' in record[\'Weights\']:\n            pair = cached  # Do not silently substitute different weights.\n        else:\n            pair = cached if all(p.is_file() for p in cached) else originals[name]\n        missing.extend(str(p) for p in pair if not p.is_file())\n        models[name] = dict(segmenter=pair[0], classifier=pair[1])\n    attention = models[\'Attention U-Net\'][\'run_dir\']\n    missing.extend(str(attention / p) for p in (\n        \'models/attention_unet_best.keras\', \'models/selected_final_classifier.keras\',\n        \'tables/selected_configuration.json\') if not (attention / p).is_file())\n    if missing:\n        raise FileNotFoundError(\'Saved checkpoints are missing. No training was started.\\n\'\n                                + \'\\n\'.join(missing))\n    return models, manifest_path.parent, manifest\n\n\ndef export_all_stages(image_path, mask_path, models, destination, previous_comparison=None,\n                      source_manifest=None):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    with Image.open(image_path) as image:\n        original = image.convert(\'RGB\').copy()\n    with Image.open(mask_path) as mask:\n        truth = np.asarray(mask.convert(\'L\').resize((256, 256), Image.Resampling.NEAREST)) > 127\n    previous = None\n    if previous_comparison and (Path(previous_comparison) / \'single_image_comparison.csv\').is_file():\n        previous = pd.read_csv(Path(previous_comparison) / \'single_image_comparison.csv\').set_index(\'Pipeline\')\n    results, rows = [], []\n    for name in ORDER:\n        tf.keras.backend.clear_session()\n        gc.collect()\n        print(\'Generating every stage:\', name, flush=True)\n        result = (detailed_attention(image_path, models[name]) if name == \'Attention U-Net\'\n                  else detailed_peer(name, image_path, models[name]))\n        result.update(name=name, **hard_overlap(truth, result[\'mask\']))\n        result[\'prediction\'] = \'Melanoma\' if result[\'score\'] >= result[\'threshold\'] else \'Benign\'\n        if previous is not None and name in previous.index:\n            for key, column in [(\'score\', \'Model score\'), (\'dice\', \'Hard Dice (this image)\'),\n                                (\'iou\', \'Hard IoU (this image)\'), (\'threshold\', \'Decision threshold\')]:\n                if not np.isclose(result[key], previous.loc[name, column], atol=5e-4, rtol=0):\n                    raise RuntimeError(f\'{name}: {key} differs from the completed comparison. \'\n                                       \'Check the checkpoint and inference configuration.\')\n        result[\'panels\'] = stage_panels(original, truth, result)\n        result[\'no_positive_cam\'] = float(np.max(result[\'heatmap\'])) <= 1e-8\n        folder = destination / \'individual_stages\' / SLUGS[name]\n        folder.mkdir(parents=True, exist_ok=True)\n        for key, value in result[\'panels\'].items():\n            if key in (\'probability\', \'heatmap\'):\n                np.save(folder / (key + \'.npy\'), value)\n                Image.fromarray(np.uint8(np.clip(value, 0, 1) * 255)).save(folder / (key + \'.png\'))\n            else:\n                Image.fromarray(value).save(folder / (key + \'.png\'))\n        np.save(folder / \'classifier_roi_input_float32.npy\', np.asarray(result[\'roi_pixels\'], np.float32))\n        np.save(folder / \'native_gradcam_heatmap.npy\', result[\'heatmap\'])\n        if result[\'full_classifier_input\'] is not None:\n            np.save(folder / \'classifier_full_input_float32.npy\', result[\'full_classifier_input\'])\n            Image.fromarray(array_rgb(result[\'full_classifier_input\'])).save(folder / \'classifier_full_input.png\')\n        rows.append({\'Pipeline\': name, \'Image ID\': Path(image_path).stem, \'True label\': \'Benign\',\n                     \'Prediction\': result[\'prediction\'], \'Model score\': result[\'score\'],\n                     \'Decision threshold\': result[\'threshold\'], \'Dice\': result[\'dice\'],\n                     \'IoU\': result[\'iou\'], \'Mask inference\': result[\'mask_method\'],\n                     \'Classifier inference\': result[\'classifier_method\'],\n                     \'ROI treatment\': result[\'roi_note\'], \'Grad-CAM target\': \'Melanoma logit\',\n                     \'No positive Grad-CAM activation\': result[\'no_positive_cam\']})\n        results.append(result)\n        tf.keras.backend.clear_session()\n        gc.collect()\n    plot_grid(results, destination, \'FULL_COMPARISON_ALL_STAGES.png\')\n    plot_grid(results, destination, \'SLIDE_A_SEGMENTATION_COMPARISON.png\', keys=KEYS[:5])\n    plot_grid(results, destination, \'SLIDE_B_ROI_AND_GRADCAM_COMPARISON.png\', keys=KEYS[5:])\n    for result in results:\n        plot_grid([result], destination, SLUGS[result[\'name\']] + \'_ALL_STAGES.png\')\n    table = pd.DataFrame(rows)\n    table.to_csv(destination / \'all_stage_predictions.csv\', index=False)\n    manifest = dict(source_manifest or {})\n    manifest.update(gradcam_target=\'Melanoma logit for every pipeline; Attention ROI branch\',\n                    heatmaps=\'Normalised separately within each image/model\',\n                    original_comparison=str(previous_comparison),\n                    checkpoint_paths={name: {k: str(v) for k, v in spec.items()}\n                                      for name, spec in models.items()},\n                    created_utc=datetime.now(timezone.utc).isoformat(),\n                    selection=\'Same selected best Attention segmentation case; true label benign\')\n    (destination / \'full_stage_provenance.json\').write_text(json.dumps(manifest, indent=2))\n    (destination / \'READ_ME.txt\').write_text(\n        \'FULL_COMPARISON_ALL_STAGES.png contains every stage for every pipeline.\\n\'\n        \'SLIDE_A and SLIDE_B split the same figure into readable presentation panels.\\n\'\n        \'Each pipeline also has its own full-stage image and a folder of separate PNGs.\\n\'\n        \'Ground truth is the expert lesion mask, not another model prediction.\\n\'\n        \'Probability maps are the segmenter output before thresholding.\\n\'\n        \'Isolated lesion is a visualisation made by multiplying the original image by \'\n        \'the predicted mask; each model retains its original classifier ROI treatment.\\n\'\n        \'Classifier ROI panels show the actual inference input, including JPEG preprocessing \'\n        \'where applicable. Exact float32 tensors are included.\\n\'\n        \'Grad-CAM explains the melanoma class in all rows. It is not a lesion mask \'\n        \'or an indication of clinical certainty. Attention shows its ROI branch; \'\n        \'that classifier also uses the full image. Its full-image input is exported separately.\\n\'\n        \'Colours use independent normalisation. A zero map has no positive activation.\\n\'\n        \'These figures concern one selected benign case, not overall pipeline performance.\\n\'\n        \'Peer models were newly trained in the previous comparison; training splits differ.\\n\'\n        \'The supplied U-Net++ notebook implements standard U-Net.\\n\')\n    package = destination / \'Melanoma_FULL_STAGE_VISUAL_COMPARISON.zip\'\n    with zipfile.ZipFile(package, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(destination.rglob(\'*\')):\n            if path.is_file() and path != package:\n                archive.write(path, path.relative_to(destination))\n    return package, table\n'}

for name, source in MODULES.items():
    (MODULE_DIR / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(MODULE_DIR))
importlib.invalidate_caches()
for name in ('attention_pipeline_core', 'bestcase_comparison_helpers', 'full_stage_visuals'):
    sys.modules.pop(name, None)
from full_stage_visuals import resolve_saved_models, export_all_stages

models, original_comparison, provenance = resolve_saved_models(
    DRIVE_ROOT, COMPARISON_ROOT, PREFERRED_COMPARISON)
print('Using checkpoints from:', original_comparison)
print('No training, threshold tuning or dataset download is needed.')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = COMPARISON_ROOT / 'full_stage_visuals' / RUN_ID
package, metrics = export_all_stages(
    CASE_DIR / 'ISIC_0000231.jpg', CASE_DIR / 'ISIC_0000231_Segmentation.png',
    models, OUTPUT_DIR, previous_comparison=original_comparison,
    source_manifest=provenance)
display(metrics.round(4))
for name in ('FULL_COMPARISON_ALL_STAGES.png', 'SLIDE_A_SEGMENTATION_COMPARISON.png',
             'SLIDE_B_ROI_AND_GRADCAM_COMPARISON.png'):
    print(name)
    display(DisplayImage(filename=str(OUTPUT_DIR / name), width=1600))
print('Finished. Figures and separate stage images:', OUTPUT_DIR)
print('Downloading the full image ZIP now.')
files.download(str(package))
